# Person 1 — Product metadata collection

**Question:** Which product characteristics are associated with average Amazon customer ratings?

Source: [McAuley Lab Amazon Reviews 2023](https://amazon-reviews-2023.github.io/),
distributed through [Hugging Face](https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023).
The overall review corpus spans May 1996–September 2023. We collect product metadata;
our subset has no review dates, so we do not claim this as its observed date range.

The four categories give us numeric variables (rating, rating count, raw price) and
categorical variables (source category, main category, store). One row is one product
entry within a category; uniqueness across categories must be checked during cleaning.
All collection logic lives in `src.collect`, not in this notebook.

In [1]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
if not (root / "src").is_dir():
    root = root.parent
if not (root / "src" / "collect.py").is_file():
    raise RuntimeError("Open this notebook from the project root or notebooks folder")
sys.path.insert(0, str(root))
from src.collect import collect_to_disk, load_raw, row_counts, REVISION
print("Source revision:", REVISION)

Source revision: 2b6d039ed471f2ba5fd2acb718bf33b0a7e5598e


C:\Users\Parker\Documents\Codex\2026-09-25\files-mentioned-by-the-user-team-2\work\miniconda3\envs\amazon-reviews\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Reproduce the full raw table

The module streams all ten parquet shards to disk to bound memory use. It caches
downloads, drops three media columns, and adds the source `category` tag. It preserves
raw values and nested types for Person 2. It neither removes duplicates nor imputes
missing values. Reading the combined table below requires more memory than the CLI.

In [2]:
output_path = collect_to_disk()
df = load_raw()
print("Saved:", output_path.relative_to(root))
print("Shape:", df.shape)

All_Beauty: 112,590 rows


Musical_Instruments: 213,593 rows


Toys_and_Games: 890,874 rows


Industrial_and_Scientific: 427,564 rows


Saved C:\Users\Parker\Documents\Codex\2026-09-25\files-mentioned-by-the-user-team\work\CAP_3764_2026_Fall_Team_6\data\raw\products_raw.parquet


Saved: data\raw\products_raw.parquet
Shape: (1644621, 14)


In [3]:
df.head()

,main_category,title,average_rating,rating_number,features,description,price,store,categories,details,parent_asin,subtitle,author,category
0,All Beauty,"Howard LC0008 Leather Conditioner, 8-Ounce (4-...",4.8,10,[],[],None,Howard Products,[],"{""Package Dimensions"": ""7.1 x 5.5 x 3 inches; ...",B01CUPMQZE,None,None,All_Beauty
1,All Beauty,Yes to Tomatoes Detoxifying Charcoal Cleanser ...,4.5,3,[],[],None,Yes To,[],"{""Item Form"": ""Powder"", ""Skin Type"": ""Acne Pro...",B076WQZGPM,None,None,All_Beauty
2,All Beauty,Eye Patch Black Adult with Tie Band (6 Per Pack),4.4,26,[],[],None,Levine Health Products,[],"{""Manufacturer"": ""Levine Health Products""}",B000B658RI,None,None,All_Beauty
3,All Beauty,"Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4...",3.1,102,[],[],None,Cherioll,[],"{""Brand"": ""Cherioll"", ""Item Form"": ""Powder"", ""...",B088FKY3VD,None,None,All_Beauty
4,All Beauty,Precision Plunger Bars for Cartridge Grips – 9...,4.3,7,"[Material: 304 Stainless Steel; Brass tip, Len...",[The Precision Plunger Bars are designed to wo...,None,Precision,[],"{""UPC"": ""644287689178""}",B07NGFDN6G,None,None,All_Beauty


In [4]:
df.dtypes

main_category      object
title              object
average_rating    float64
rating_number       int64
features           object
description        object
price              object
store              object
categories         object
details            object
parent_asin        object
subtitle           object
author             object
category           object
dtype: object

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1644621 entries, 0 to 1644620
Data columns (total 14 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   main_category   1599089 non-null  object 
 1   title           1644621 non-null  object 
 2   average_rating  1644621 non-null  float64
 3   rating_number   1644621 non-null  int64  
 4   features        1644621 non-null  object 
 5   description     1644621 non-null  object 
 6   price           1644621 non-null  object 
 7   store           1620925 non-null  object 
 8   categories      1644621 non-null  object 
 9   details         1644621 non-null  object 
 10  parent_asin     1644621 non-null  object 
 11  subtitle        445 non-null      object 
 12  author          173 non-null      object 
 13  category        1644621 non-null  object 
dtypes: float64(1), int64(1), object(12)
memory usage: 175.7+ MB


In [6]:
counts = df["category"].value_counts().sort_index()
counts.rename("rows").to_frame()

,rows
category,
All_Beauty,112590
Industrial_and_Scientific,427564
Musical_Instruments,213593
Toys_and_Games,890874


In [7]:
expected = row_counts()
assert counts.to_dict() == dict(sorted(expected.items()))
assert not {"images", "videos", "bought_together"}.intersection(df.columns)
assert {"parent_asin", "average_rating", "rating_number", "price", "main_category", "store", "category"}.issubset(df.columns)
print("All source rows accounted for:", sum(expected.values()))

All source rows accounted for: 1644621


## Handoff to Person 2

Run `python -m src.collect` from the repository root, then use
`from src.collect import load_raw; df = load_raw()` in your cleaning notebook.
The file is `data/raw/products_raw.parquet` and is intentionally excluded from Git.

Inspect the actual raw price and category representations before converting them.
Use `parent_asin` to investigate duplicates; distinguish the source `category` tag
from `main_category`. A store name is not necessarily a brand. Missing values and
duplicate entries have deliberately not been altered.

This notebook verifies collection, not statistical findings. Person 2 owns cleaning,
Person 3 owns summary statistics, and Person 4 owns charts. Price/rating associations
cannot by themselves establish causation.